# 🚀 StyleMatch AI - Model Server (Colab Side)

This notebook hosts the CLIP model and provides an API for your local Docker backend.

### Steps:
1. Enable GPU: `Runtime` -> `Change runtime type` -> `T4 GPU`.
2. Run all cells.
3. Copy the **ngrok public URL** and paste it into your local `docker-compose.yml`.

In [ ]:
# 1. Install dependencies
!pip install torch transformers pillow pandas numpy flask pyngrok

In [ ]:
# 2. Configuration (Change your token here)
NGROK_TOKEN = "YOUR_NGROK_AUTHTOKEN"
!ngrok config add-authtoken {NGROK_TOKEN}

In [ ]:
# 3. Check/Create app_colab.py
import os
if not os.path.exists('app_colab.py'):
    print("⚠️ File app_colab.py NOT found! Please upload it to Colab.")
else:
    print("✅ app_colab.py is ready.")

In [ ]:
# 4. Start Server and Ngrok
from pyngrok import ngrok
import threading
import time
import requests

DISCORD_WEBHOOK = "https://discord.com/api/webhooks/1491031437288280146/B5xnLJ_qxU_nxL63GnzUmSvhCHGMw9Dxv5SzxQLyBJVs0V-Y2Jf22y2svv6DLiqPRti5"

def send_discord(content=None, embed=None):
    if not DISCORD_WEBHOOK: return
    payload = {}
    if content: payload['content'] = content
    if embed: payload['embeds'] = [embed]
    try: requests.post(DISCORD_WEBHOOK, json=payload, timeout=5)
    except: pass

# Kill old sessions
ngrok.kill()

# Start Flask in background
def run_flask():
    !python app_colab.py

threading.Thread(target=run_flask, daemon=True).start()
time.sleep(10) # Wait for model to load

# Open tunnel
try:
    public_url = ngrok.connect(5000).public_url
    print("\n" + "="*50)
    print(f"🌍 COLAB SERVER URL: {public_url}")
    print("="*50)
    print("Copy this URL to your local docker-compose.yml")

    # Notify Discord
    send_discord(embed={
        'title': '🚀 StyleMatch AI Colab — Started',
        'color': 0x2ECC71,
        'fields': [
            {'name': '🌐 URL', 'value': public_url, 'inline': False},
            {'name': '🖥️ Status', 'value': 'Online', 'inline': True}
        ]
    })
    print("✅ Sent Discord notification!")
except Exception as e:
    print(f"❌ Ngrok error: {e}")